In [ ]:
# ============================================================
# CLEAN ENVIRONMENT — REMOVE bitsandbytes + peft completely
# ============================================================

!pip uninstall -y bitsandbytes
!pip uninstall -y peft

# Install clean stack (no bitsandbytes, no peft)
!pip install -q torch==2.1.2
!pip install -q transformers==4.38.2
!pip install -q datasets==2.17.1
!pip install -q accelerate==0.27.2
!pip install -q evaluate==0.4.1
!pip install -q trl==0.7.11
!pip install -q sentencepiece

print("✅ Clean environment ready. PLEASE RESTART RUNTIME now.")

Found existing installation: bitsandbytes 0.43.0
Uninstalling bitsandbytes-0.43.0:
  Successfully uninstalled bitsandbytes-0.43.0
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.1.2
✅ Clean environment ready. PLEASE RESTART RUNTIME now.


In [ ]:
# ============================================================
# CELL 2 — Imports, Reproducibility, and Device Setup
# ============================================================
# This cell:
# 1) Imports all required libraries for Classic RLHF.
# 2) Sets deterministic seeds.
# 3) Detects CUDA availability.
# 4) Confirms that bitsandbytes is NOT installed (sanity check).

import os
import gc
import random
import numpy as np
import torch

# Hugging Face
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)

# TRL (Classic PPO pipeline components)
from trl import (
    SFTTrainer,
    PPOTrainer,
    PPOConfig,
    AutoModelForCausalLMWithValueHead,
    create_reference_model,
)

# ============================================================
# Reproducibility
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# Device setup
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

# ============================================================
# Memory cleanup helper (important for Colab stability)
# ============================================================

def cleanup():
    """
    Frees unused memory to reduce risk of CUDA OOM
    between SFT, Reward Model, and PPO stages.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# Safety check: ensure bitsandbytes is NOT loaded
# ============================================================

try:
    import bitsandbytes
    print("⚠️ bitsandbytes is still installed. This may cause CUDA errors.")
except ImportError:
    print("✅ bitsandbytes not installed (good).")

print("✅ Cell 2 complete. Ready for Cell 3.")

Device: cuda
GPU: Tesla T4
CUDA version: 12.8
✅ bitsandbytes not installed (good).
✅ Cell 2 complete. Ready for Cell 3.


In [ ]:
# ============================================================
# CELL 3 — Base Model Configuration + Directory Setup
# ============================================================
# This cell:
# 1) Defines which base model we will use for SFT → RM → PPO.
# 2) Creates directories to store checkpoints.
# 3) Sets global training hyperparameters for later cells.
#
# We use a small GPT2 variant for stability on Colab GPU.

# ------------------------------------------------------------
# Choose base model
# ------------------------------------------------------------
# distilgpt2 is lightweight and fits comfortably on T4 GPUs.
# You can switch to "gpt2" later if you want better quality.
BASE_MODEL = "distilgpt2"

# ------------------------------------------------------------
# Directory structure
# ------------------------------------------------------------
ROOT_DIR = "/content/rlhf_classic"

SFT_DIR = os.path.join(ROOT_DIR, "sft_policy")
RM_DIR  = os.path.join(ROOT_DIR, "reward_model")
PPO_DIR = os.path.join(ROOT_DIR, "ppo_policy")

os.makedirs(SFT_DIR, exist_ok=True)
os.makedirs(RM_DIR, exist_ok=True)
os.makedirs(PPO_DIR, exist_ok=True)

print("Checkpoint directories created:")
print("SFT_DIR:", SFT_DIR)
print("RM_DIR :", RM_DIR)
print("PPO_DIR:", PPO_DIR)

# ------------------------------------------------------------
# Sequence length limits (kept small for Colab stability)
# ------------------------------------------------------------
MAX_SFT_LEN = 256
MAX_RM_LEN  = 256

# PPO generation parameters (used later)
MAX_NEW_TOKENS = 64

print("\nBase model:", BASE_MODEL)
print("Max SFT length:", MAX_SFT_LEN)
print("Max RM length :", MAX_RM_LEN)
print("Max PPO new tokens:", MAX_NEW_TOKENS)

print("\n✅ Cell 3 complete. Ready for Cell 4.")

Checkpoint directories created:
SFT_DIR: /content/rlhf_classic/sft_policy
RM_DIR : /content/rlhf_classic/reward_model
PPO_DIR: /content/rlhf_classic/ppo_policy

Base model: distilgpt2
Max SFT length: 256
Max RM length : 256
Max PPO new tokens: 64

✅ Cell 3 complete. Ready for Cell 4.


In [ ]:
# ============================================================
# CELL 4 — Build Supervised Fine-Tuning (SFT) Dataset
# ============================================================
# This cell:
# 1) Creates a small instruction-response dataset.
# 2) Formats it into a causal LM training style.
# 3) Converts it into a HuggingFace Dataset object.
#
# This is the first stage of classic RLHF: Supervised Fine-Tuning (SFT).
# In real RLHF you would replace this with a large instruction dataset.

# ------------------------------------------------------------
# Step 1: Define instruction–response pairs
# ------------------------------------------------------------

sft_pairs = [
    ("Explain entropy in simple terms.",
     "Entropy measures how spread out or uncertain energy or information is. Higher entropy generally means more disorder or unpredictability."),

    ("Write a polite reminder for a meeting tomorrow.",
     "Hi! Just a friendly reminder about our meeting tomorrow. Please let me know if anything changes. Looking forward to it."),

    ("Give three tips to improve focus while studying.",
     "1) Use focused time blocks like 25-minute sessions. 2) Keep your phone away. 3) Study in a quiet, consistent environment."),

    ("Explain Newton's first law in simple words.",
     "An object will keep doing what it is doing—staying still or moving steadily—unless a force acts on it."),

    ("What is recursion in programming?",
     "Recursion is when a function solves a problem by calling itself on smaller versions of the same problem until a base case is reached."),
]

# ------------------------------------------------------------
# Step 2: Format into causal LM style
# ------------------------------------------------------------
# We use a simple instruction template.
# The model will learn to generate the Response after the Instruction.

def format_sft(prompt, answer):
    return f"### Instruction:\n{prompt}\n\n### Response:\n{answer}\n"

formatted_texts = [format_sft(p, a) for p, a in sft_pairs]

# ------------------------------------------------------------
# Step 3: Create HuggingFace Dataset
# ------------------------------------------------------------

sft_dataset = Dataset.from_dict({"text": formatted_texts})

print("SFT dataset size:", len(sft_dataset))
print("\nExample formatted training sample:\n")
print(sft_dataset[0]["text"])

print("\n✅ Cell 4 complete. Ready for Cell 5.")

SFT dataset size: 5

Example formatted training sample:

### Instruction:
Explain entropy in simple terms.

### Response:
Entropy measures how spread out or uncertain energy or information is. Higher entropy generally means more disorder or unpredictability.


✅ Cell 4 complete. Ready for Cell 5.


In [ ]:
# ============================================================
# CELL 5 — Load Tokenizer and Base Language Model (for SFT)
# ============================================================
# This cell:
# 1) Loads the tokenizer for the base model.
# 2) Ensures a pad_token exists (GPT2-family does not define one by default).
# 3) Loads the base causal LM onto the correct device.
#
# This model will be fine-tuned in the SFT stage.

# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# GPT2 models do not define a pad token by default.
# For training with batching, we set pad_token = eos_token.
if tokenizer.pad_token is None:
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

# ------------------------------------------------------------
# Load base causal language model
# ------------------------------------------------------------

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

# Move model to GPU if available
base_model = base_model.to(device)

print("\nBase model loaded and moved to device:", device)

# Quick parameter count check (sanity)
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Total parameters: {total_params:,}")

print("\n✅ Cell 5 complete. Ready for Cell 6.")

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|endoftext|>

Base model loaded and moved to device: cuda
Total parameters: 81,912,576

✅ Cell 5 complete. Ready for Cell 6.


In [ ]:
# ============================================================
# CELL 6 — Supervised Fine-Tuning (SFT) Training
# ============================================================
# This cell performs Stage 1 of classic RLHF:
#     Pretraining → SFT (instruction tuning)
#
# We fine-tune the base causal LM on the instruction-response dataset
# created in Cell 4.
#
# Keep training light for Colab stability. You can increase epochs later.

from transformers import TrainingArguments

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

sft_training_args = TrainingArguments(
    output_dir=SFT_DIR,
    per_device_train_batch_size=2,     # small batch for GPU stability
    gradient_accumulation_steps=1,
    num_train_epochs=1,                # increase for stronger SFT
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    fp16=(device == "cuda"),           # use mixed precision only if CUDA
    report_to=[],
)

# ------------------------------------------------------------
# Initialize SFTTrainer
# ------------------------------------------------------------

sft_trainer = SFTTrainer(
    model=base_model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SFT_LEN,
    args=sft_training_args,
)

print("Starting SFT training...\n")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

sft_trainer.train()

# ------------------------------------------------------------
# Save SFT policy checkpoint
# ------------------------------------------------------------

sft_trainer.model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)

print("\n✅ SFT training complete.")
print("Saved SFT model to:", SFT_DIR)

# Free memory before moving to reward model stage
cleanup()

print("\n✅ Cell 6 complete. Ready for Cell 7.")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:294: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


Starting SFT training...



Step,Training Loss
1,6.061300
2,6.329400
3,3.949100



✅ SFT training complete.
Saved SFT model to: /content/rlhf_classic/sft_policy

✅ Cell 6 complete. Ready for Cell 7.


In [ ]:
# ============================================================
# CELL 7 — Build Preference Dataset for Reward Model
# ============================================================
# This cell creates the dataset used to train the reward model.
#
# Classic RLHF Stage 2:
#   Human (or synthetic) preference pairs:
#       (prompt + chosen_answer)   → higher reward
#       (prompt + rejected_answer) → lower reward
#
# In real RLHF, these come from human comparisons.
# Here we create synthetic rejected answers for demonstration.

# ------------------------------------------------------------
# Step 1: Define low-quality (rejected) responses
# ------------------------------------------------------------

bad_responses = [
    "I don't know.",
    "No idea.",
    "Maybe.",
    "This is difficult.",
    "I cannot answer that.",
]

# ------------------------------------------------------------
# Step 2: Build chosen and rejected examples
# ------------------------------------------------------------

chosen_texts = []
rejected_texts = []

for (prompt, good_answer) in sft_pairs:
    formatted_good = format_sft(prompt, good_answer)
    formatted_bad = format_sft(prompt, random.choice(bad_responses))

    chosen_texts.append(formatted_good)
    rejected_texts.append(formatted_bad)

# ------------------------------------------------------------
# Step 3: Create HuggingFace Dataset
# ------------------------------------------------------------

reward_dataset = Dataset.from_dict({
    "chosen": chosen_texts,
    "rejected": rejected_texts,
})

print("Reward dataset size:", len(reward_dataset))
print("\nExample preference pair:\n")
print("CHOSEN:\n", reward_dataset[0]["chosen"])
print("\nREJECTED:\n", reward_dataset[0]["rejected"])

print("\n✅ Cell 7 complete. Ready for Cell 8.")

Reward dataset size: 5

Example preference pair:

CHOSEN:
 ### Instruction:
Explain entropy in simple terms.

### Response:
Entropy measures how spread out or uncertain energy or information is. Higher entropy generally means more disorder or unpredictability.


REJECTED:
 ### Instruction:
Explain entropy in simple terms.

### Response:
No idea.


✅ Cell 7 complete. Ready for Cell 8.


In [ ]:
# ============================================================
# CELL 8 — Tokenize Preference Pairs for Reward Model Training
# ============================================================
# This cell:
# 1) Tokenizes the chosen and rejected texts separately.
# 2) Pads/truncates to MAX_RM_LEN for stable batching.
# 3) Produces tensor-ready columns for pairwise reward training.
#
# Output columns:
#   - input_ids_chosen
#   - attention_mask_chosen
#   - input_ids_rejected
#   - attention_mask_rejected

# ------------------------------------------------------------
# Tokenization function
# ------------------------------------------------------------

def tokenize_reward_pair(example):
    chosen_tokens = tokenizer(
        example["chosen"],
        truncation=True,
        padding="max_length",
        max_length=MAX_RM_LEN,
    )

    rejected_tokens = tokenizer(
        example["rejected"],
        truncation=True,
        padding="max_length",
        max_length=MAX_RM_LEN,
    )

    return {
        "input_ids_chosen": chosen_tokens["input_ids"],
        "attention_mask_chosen": chosen_tokens["attention_mask"],
        "input_ids_rejected": rejected_tokens["input_ids"],
        "attention_mask_rejected": rejected_tokens["attention_mask"],
    }

# ------------------------------------------------------------
# Apply tokenization
# ------------------------------------------------------------

reward_tokenized = reward_dataset.map(
    tokenize_reward_pair,
    remove_columns=reward_dataset.column_names,
)

print("Tokenized reward dataset columns:")
print(reward_tokenized.column_names)

print("\nExample tokenized pair shapes:")
print("Chosen length:", len(reward_tokenized[0]["input_ids_chosen"]))
print("Rejected length:", len(reward_tokenized[0]["input_ids_rejected"]))

print("\n✅ Cell 8 complete. Ready for Cell 9.")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenized reward dataset columns:
['input_ids_chosen', 'attention_mask_chosen', 'input_ids_rejected', 'attention_mask_rejected']

Example tokenized pair shapes:
Chosen length: 256
Rejected length: 256

✅ Cell 8 complete. Ready for Cell 9.


In [ ]:
# ============================================================
# CELL 9 — Train Reward Model (GPT2 Padding)
# ============================================================
# Fix:
# GPT2ForSequenceClassification requires pad_token_id defined
# when batch_size > 1.
#
# We explicitly set:
#   reward_model.config.pad_token_id = tokenizer.eos_token_id

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# 1️⃣ Initialize reward model
# ------------------------------------------------------------

reward_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
).to(device)

# 🔥 IMPORTANT FIX
reward_model.config.pad_token_id = tokenizer.eos_token_id

reward_model.train()
print("Reward model initialized (pad_token_id fixed).")

# ------------------------------------------------------------
# 2️⃣ DataLoader
# ------------------------------------------------------------

def reward_collate(batch):
    def stack(key):
        return torch.tensor(
            [example[key] for example in batch],
            dtype=torch.long,
            device=device
        )

    return {
        "input_ids_chosen": stack("input_ids_chosen"),
        "attention_mask_chosen": stack("attention_mask_chosen"),
        "input_ids_rejected": stack("input_ids_rejected"),
        "attention_mask_rejected": stack("attention_mask_rejected"),
    }

reward_loader = DataLoader(
    reward_tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=reward_collate,
)

# ------------------------------------------------------------
# 3️⃣ Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=2e-5)

# ------------------------------------------------------------
# 4️⃣ Pairwise training loop
# ------------------------------------------------------------

epochs = 2

for epoch in range(epochs):
    losses = []

    for batch in reward_loader:
        optimizer.zero_grad()

        r_chosen = reward_model(
            input_ids=batch["input_ids_chosen"],
            attention_mask=batch["attention_mask_chosen"],
        ).logits.squeeze(-1)

        r_rejected = reward_model(
            input_ids=batch["input_ids_rejected"],
            attention_mask=batch["attention_mask_rejected"],
        ).logits.squeeze(-1)

        loss = -torch.nn.functional.logsigmoid(r_chosen - r_rejected).mean()

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{epochs} | Reward loss: {np.mean(losses):.4f}")

# ------------------------------------------------------------
# 5️⃣ Save reward model
# ------------------------------------------------------------

reward_model.save_pretrained(RM_DIR)
tokenizer.save_pretrained(RM_DIR)

print("\n✅ Reward model training complete.")
print("Saved to:", RM_DIR)

cleanup()
print("\n✅ Cell 9 fixed and complete. Ready for Cell 10.")

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at distilgpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Reward model initialized (pad_token_id fixed).
Epoch 1/2 | Reward loss: 0.9271
Epoch 2/2 | Reward loss: 0.9079

✅ Reward model training complete.
Saved to: /content/rlhf_classic/reward_model

✅ Cell 9 fixed and complete. Ready for Cell 10.


In [ ]:
# ============================================================
# CELL 10 — PPO Setup (FIXED DATASET COLLATION)
# ============================================================
# Fix:
# - Do NOT manually encode input_ids.
# - Keep dataset as {"query": text}.
# - Let PPOTrainer handle tokenization & padding.

cleanup()

# ------------------------------------------------------------
# 1️⃣ Load SFT policy with value head
# ------------------------------------------------------------

policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(SFT_DIR)
policy_model = policy_model.to(device)
policy_model.config.pad_token_id = tokenizer.eos_token_id

print("Policy model loaded.")

# ------------------------------------------------------------
# 2️⃣ Create frozen reference model
# ------------------------------------------------------------

ref_model = create_reference_model(policy_model)

print("Reference model created.")

# ------------------------------------------------------------
# 3️⃣ PPO dataset (RAW TEXT ONLY)
# ------------------------------------------------------------

ppo_prompts = [
    f"### Instruction:\n{prompt}\n\n### Response:\n"
    for (prompt, _) in sft_pairs
]

ppo_dataset = Dataset.from_dict({"query": ppo_prompts})

print("PPO dataset ready.")
print("Example:", ppo_dataset[0]["query"])

# ------------------------------------------------------------
# 4️⃣ PPO config
# ------------------------------------------------------------

ppo_config = PPOConfig(
    model_name=BASE_MODEL,
    learning_rate=1e-5,
    batch_size=2,
    mini_batch_size=1,
    ppo_epochs=1,
    log_with=None,
)

# ------------------------------------------------------------
# 5️⃣ Initialize PPOTrainer
# ------------------------------------------------------------

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
)

print("\n✅ PPOTrainer initialized correctly.")
print("Ready for Cell 11.")

Policy model loaded.
Reference model created.
PPO dataset ready.
Example: ### Instruction:
Explain entropy in simple terms.

### Response:


✅ PPOTrainer initialized correctly.
Ready for Cell 11.


In [ ]:
# ============================================================
# CELL 11 — Reward Scoring Function (Fixed for PPO Setup)
# ============================================================
# This loads the trained reward model and defines
# a stable reward scoring function for PPO.
#
# Input: list of full texts (prompt + generated response)
# Output: tensor of scalar rewards

# ------------------------------------------------------------
# Load trained reward model
# ------------------------------------------------------------

reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_DIR,
    num_labels=1,
).to(device)

reward_model.config.pad_token_id = tokenizer.eos_token_id
reward_model.eval()

print("Reward model loaded for PPO scoring.")

# ------------------------------------------------------------
# Reward function
# ------------------------------------------------------------

@torch.no_grad()
def compute_reward(text_list):
    """
    text_list: list[str]
    returns: torch.FloatTensor (batch_size,)
    """

    tokens = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=MAX_RM_LEN,
        return_tensors="pt"
    ).to(device)

    outputs = reward_model(**tokens)
    rewards = outputs.logits.squeeze(-1)

    return rewards.detach().cpu()

print("Reward function ready.")
print("\n✅ Cell 11 fixed. Ready for Cell 12.")

Reward model loaded for PPO scoring.
Reward function ready.

✅ Cell 11 fixed. Ready for Cell 12.


In [ ]:
# ============================================================
# CELL 12 — PPO Training Loop (FINAL CORRECT VERSION)
# ============================================================
# TRL 0.7 requires:
#   generate() expects List[Tensor]
#   step() expects List[Tensor]
#
# So we:
#   1) Tokenize queries manually
#   2) Pass token tensors
#   3) Decode for reward
#   4) PPO update

generation_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    top_p=0.95,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id,
)

policy_model.train()

print("Starting PPO training...\n")

for batch in ppo_trainer.dataloader:

    # --------------------------------------------------------
    # 1️⃣ Manually tokenize queries into tensors
    # --------------------------------------------------------

    query_texts = batch["query"]

    tokenized = tokenizer(
        query_texts,
        padding=True,
        return_tensors="pt"
    ).to(device)

    query_tensors = [
        tokenized["input_ids"][i]
        for i in range(tokenized["input_ids"].shape[0])
    ]

    # --------------------------------------------------------
    # 2️⃣ Generate responses
    # --------------------------------------------------------

    response_tensors = ppo_trainer.generate(
        query_tensors,
        **generation_kwargs
    )

    # --------------------------------------------------------
    # 3️⃣ Decode responses for reward scoring
    # --------------------------------------------------------

    responses = [
        tokenizer.decode(r.squeeze(), skip_special_tokens=True)
        for r in response_tensors
    ]

    full_texts = [
        q + resp for q, resp in zip(query_texts, responses)
    ]

    rewards = compute_reward(full_texts)

    reward_tensors = [
        torch.tensor(r, dtype=torch.float32)
        for r in rewards
    ]

    # --------------------------------------------------------
    # 4️⃣ PPO update
    # --------------------------------------------------------

    stats = ppo_trainer.step(
        query_tensors,
        response_tensors,
        reward_tensors,
    )

    ppo_trainer.log_stats(stats, batch, reward_tensors)

print("\n✅ PPO training complete.")

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Starting PPO training...



/tmp/ipython-input-3284192907.py:70: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(r, dtype=torch.float32)



✅ PPO training complete.


In [ ]:
# ============================================================
# CELL 13 — Save PPO Policy + Compare Outputs
# ============================================================
# This cell:
# 1) Saves the PPO-updated model.
# 2) Compares SFT vs PPO outputs on a new prompt.
# 3) Lets you see qualitative reward alignment change.

# ------------------------------------------------------------
# 1️⃣ Save PPO model
# ------------------------------------------------------------

ppo_trainer.model.save_pretrained(PPO_DIR)
tokenizer.save_pretrained(PPO_DIR)

print("✅ PPO policy saved to:", PPO_DIR)

# ------------------------------------------------------------
# 2️⃣ Utility function to generate from saved model
# ------------------------------------------------------------

@torch.no_grad()
def generate_from_model(model_dir, prompt):
    tok = AutoTokenizer.from_pretrained(model_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_dir).to(device)
    model.config.pad_token_id = tok.eos_token_id

    inputs = tok(prompt, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tok.eos_token_id,
    )

    return tok.decode(output[0], skip_special_tokens=True)


# ------------------------------------------------------------
# 3️⃣ Compare SFT vs PPO
# ------------------------------------------------------------

test_prompt = "### Instruction:\nExplain gradient descent in simple terms.\n\n### Response:\n"

print("\n===== SFT OUTPUT =====\n")
print(generate_from_model(SFT_DIR, test_prompt))

print("\n===== PPO OUTPUT =====\n")
print(generate_from_model(PPO_DIR, test_prompt))

print("\n✅ Classic RLHF pipeline complete.")

✅ PPO policy saved to: /content/rlhf_classic/ppo_policy

===== SFT OUTPUT =====



Some weights of the model checkpoint at /content/rlhf_classic/ppo_policy were not used when initializing GPT2LMHeadModel: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Instruction:
Explain gradient descent in simple terms.

### Response:
This tutorial is based on a recent blog post, which is a bit confusing. I wanted to try to highlight the effect of gradient descent in a better way.

===== PPO OUTPUT =====

### Instruction:
Explain gradient descent in simple terms.

### Response:
The gradient descent is a new way of thinking about gradient descent. We've written a simple problem that uses a gradient descent to map an element in the gradient, so that it can be called gradient descent. The gradient descent is a new way of thinking about gradient descent.
The gradient descent is a new way of thinking about gradient descent. We've written a simple problem that uses a gradient descent to map an element in the gradient, so that it can be called gradient descent. The gradient descent is

✅ Classic RLHF pipeline complete.


# ============================================
# CELL 14 — Mathematical View of PPO in RLHF
# ============================================
# In RLHF, we optimize:
#
#     π_θ  ←  argmax  E[ r(x, y) - β KL( π_θ || π_ref ) ]
#
# where:
#   x = prompt
#   y = generated response
#   r(x, y) = reward from reward model
#   β = KL penalty coefficient
#
# PPO uses clipped objective:
#
#     L = E[ min( r_t(θ) A_t ,
#                 clip(r_t(θ), 1-ε, 1+ε) A_t ) ]
#
# where:
#   r_t(θ) = π_θ(a_t|s_t) / π_old(a_t|s_t)
#   A_t = advantage estimate
#
# In TRL:
#   - Advantage = reward - value_head_prediction
#   - KL penalty prevents divergence from SFT model


In [ ]:
# ============================================================
# CELL 14 — Proper KL Divergence (Final Correct Version)
# ============================================================
# Both policy_model and ref_model are wrapped by TRL.
# Their forward() returns a tuple:
#
#   outputs = (logits, _, values)
#
# So we must always unpack index [0] for logits.

import torch.nn.functional as F

@torch.no_grad()
def compute_kl(query_text):
    tokens = tokenizer(query_text, return_tensors="pt").to(device)

    # Policy forward pass
    policy_outputs = policy_model(**tokens)
    policy_logits = policy_outputs[0]

    # Reference forward pass
    ref_outputs = ref_model(**tokens)
    ref_logits = ref_outputs[0]

    # Log probabilities
    policy_logprob = F.log_softmax(policy_logits, dim=-1)
    ref_logprob = F.log_softmax(ref_logits, dim=-1)

    # KL estimate
    kl = (policy_logprob - ref_logprob).mean()

    return kl.item()

sample_prompt = ppo_prompts[0]
print("KL divergence example:", compute_kl(sample_prompt))

KL divergence example: 0.3560643196105957


In [ ]:
# ============================================================
# CELL 15 — Token-Level KL on Generated Response
# ============================================================
# This computes KL only over generated tokens.
#
# True RLHF KL:
#
#   KL = Σ_t [ log π_policy(y_t | x,y_<t)
#              - log π_ref(y_t | x,y_<t) ]
#
# Computed only for response tokens (not prompt tokens).
#
# This matches what PPOTrainer internally penalizes.

import torch.nn.functional as F

@torch.no_grad()
def compute_response_kl(prompt_text, max_new_tokens=50):

    # Tokenize prompt
    prompt_tokens = tokenizer(prompt_text, return_tensors="pt").to(device)

    # Generate response
    output = policy_model.generate(
        **prompt_tokens,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_ids = output[0]

    # Split prompt and response
    prompt_len = prompt_tokens["input_ids"].shape[1]
    response_ids = full_ids[prompt_len:]

    # Forward full sequence through both models
    full_tokens = tokenizer(
        tokenizer.decode(full_ids, skip_special_tokens=True),
        return_tensors="pt"
    ).to(device)

    policy_logits = policy_model(**full_tokens)[0]
    ref_logits = ref_model(**full_tokens)[0]

    policy_logprob = F.log_softmax(policy_logits, dim=-1)
    ref_logprob = F.log_softmax(ref_logits, dim=-1)

    # Compute KL only on response tokens
    kl_values = []

    for t in range(prompt_len, full_tokens["input_ids"].shape[1]):
        token_id = full_tokens["input_ids"][0, t]
        kl_t = policy_logprob[0, t, token_id] - ref_logprob[0, t, token_id]
        kl_values.append(kl_t.item())

    return sum(kl_values) / len(kl_values)

sample_prompt = ppo_prompts[0]
print("Token-level response KL:", compute_response_kl(sample_prompt))

Token-level response KL: 0.3910511493682861


In [ ]:
# ============================================================
# CELL 16 — Reward vs KL Tradeoff (r - β·KL) on Multiple Prompts
# ============================================================
# RLHF objective (one-step view):
#
#   J(θ) = E_{y~πθ}[ r(x,y) - β * KL(πθ || πref) ]
#
# For a batch of prompts {x_i}, we can estimate:
#   - reward_i = r(x_i, y_i)
#   - kl_i     = token-level KL over generated y_i
#   - obj_i    = reward_i - β * kl_i
#
# This cell:
# 1) Generates one response per prompt.
# 2) Computes reward_i via reward model.
# 3) Computes kl_i via response-only token KL.
# 4) Prints summary statistics.
#
# NOTE: This is an *analysis* diagnostic; PPO already did training.

import numpy as np

BETA = 0.1  # β controls how strongly we penalize divergence from reference

def generate_one_response(prompt_text, max_new_tokens=60):
    """Generate a single response continuation from the PPO policy."""
    tokens = tokenizer(prompt_text, return_tensors="pt").to(device)
    out = policy_model.generate(
        **tokens,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# ------------------------------------------------------------
# Evaluate a small batch of prompts
# ------------------------------------------------------------

eval_prompts = ppo_prompts[:]  # you can slice, e.g., ppo_prompts[:5] if you want faster

rewards = []
kls = []
objs = []

for p in eval_prompts:
    # Generate full text (prompt + response)
    full_text = generate_one_response(p, max_new_tokens=MAX_NEW_TOKENS)

    # Reward: scalar r(x,y)
    r = compute_reward([full_text])[0].item()

    # KL: response-only token KL
    kl = compute_response_kl(p, max_new_tokens=MAX_NEW_TOKENS)

    # RLHF objective proxy
    obj = r - BETA * kl

    rewards.append(r)
    kls.append(kl)
    objs.append(obj)

print("===== Reward/KL Summary =====")
print("Reward mean:", float(np.mean(rewards)), "| std:", float(np.std(rewards)))
print("KL mean    :", float(np.mean(kls)),     "| std:", float(np.std(kls)))
print("Obj mean   :", float(np.mean(objs)),    "| std:", float(np.std(objs)))

# Show a few samples
print("\n===== First 3 Samples =====")
for i in range(min(3, len(eval_prompts))):
    print("\n--- Sample", i+1, "---")
    print("Reward:", rewards[i])
    print("KL    :", kls[i])
    print("Obj   :", objs[i])

===== Reward/KL Summary =====
Reward mean: -2.9902530670166017 | std: 0.9189428453414449
KL mean    : 0.2964957930147648 | std: 0.06409578192290541
Obj mean   : -3.019902646318078 | std: 0.9206446827097775

===== First 3 Samples =====

--- Sample 1 ---
Reward: -3.6992099285125732
KL    : 0.29603803902864456
Obj   : -3.728813732415438

--- Sample 2 ---
Reward: -4.2612104415893555
KL    : 0.3133080303668976
Obj   : -4.292541244626046

--- Sample 3 ---
Reward: -1.6351678371429443
KL    : 0.21875184774398804
Obj   : -1.657043021917343


In [ ]:
# ============================================================
# CELL 17 — PPO Ratio and Clipping Behavior
# ============================================================
# PPO core mechanism:
#
#   r_t(θ) = π_θ(a_t|s_t) / π_old(a_t|s_t)
#
# Objective:
#
#   L = E[ min( r_t A_t ,
#               clip(r_t, 1-ε, 1+ε) A_t ) ]
#
# Clipping prevents large destructive policy updates.
#
# If:
#   r_t > 1+ε  → clipped
#   r_t < 1-ε  → clipped
#
# This cell simulates the ratio behavior numerically.

import numpy as np

epsilon = 0.2  # typical PPO clip range

# Simulate policy probability change
old_probs = np.array([0.2, 0.3, 0.5])
new_probs = np.array([0.4, 0.1, 0.5])  # some large change

ratios = new_probs / old_probs

clipped_ratios = np.clip(ratios, 1 - epsilon, 1 + epsilon)

advantages = np.array([1.0, 0.5, -0.3])

unclipped_obj = ratios * advantages
clipped_obj = clipped_ratios * advantages

print("Old probs     :", old_probs)
print("New probs     :", new_probs)
print("Ratios        :", ratios)
print("Clipped ratios:", clipped_ratios)

print("\nUnclipped objective:", unclipped_obj)
print("Clipped objective  :", clipped_obj)

print("\nFinal PPO objective (min of both):")
print(np.minimum(unclipped_obj, clipped_obj))

Old probs     : [0.2 0.3 0.5]
New probs     : [0.4 0.1 0.5]
Ratios        : [2.         0.33333333 1.        ]
Clipped ratios: [1.2 0.8 1. ]

Unclipped objective: [ 2.          0.16666667 -0.3       ]
Clipped objective  : [ 1.2  0.4 -0.3]

Final PPO objective (min of both):
[ 1.2         0.16666667 -0.3       ]
